# Scenario Schema Consistency Test

Run Scenario Schema **3 times on the same origin date** and compare:
- Scenario design and structure
- Price point forecasts
- Full price distributions (quantiles)
- Rationales

This reveals whether the agent's estimates are stable or show significant variance across runs.

## Setup

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from aieng.forecasting.models import LITE_MODEL
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_multivariate_service
from energy_oil_forecasting.analyst_agent import (
    build_wti_news_scenario_schema_config,
    build_wti_scenario_schema_predictor,
)

# Configuration
ORIGIN_DATE = pd.Timestamp("2026-05-25")  # Change this to test different dates
NUM_RUNS = 3
HORIZONS = [5, 10, 21]

# Setup
data_service = build_wti_multivariate_service()
print(f"Testing Scenario Schema on origin: {ORIGIN_DATE.date()}")
print(f"Horizons: {HORIZONS} business days")
print(f"Number of runs: {NUM_RUNS}")

Testing Scenario Schema on origin: 2026-05-25
Horizons: [5, 10, 21] business days
Number of runs: 3


## Run Scenario Schema 3 Times

In [2]:
from aieng.forecasting.evaluation import MultiTargetBacktestSpec, cached_multi_backtest

results = []

for run_num in range(NUM_RUNS):
    print(f"\n{'='*72}")
    print(f"RUN {run_num + 1} / {NUM_RUNS}")
    print(f"{'='*72}")
    
    # Build predictor
    config = build_wti_news_scenario_schema_config(model=LITE_MODEL)
    predictor = build_wti_scenario_schema_predictor(config)
    
    # Create a minimal spec for just this one origin
    spec = MultiTargetBacktestSpec.model_validate({
        "spec_id": f"consistency_test_run_{run_num}",
        "start": ORIGIN_DATE.strftime("%Y-%m-%d"),
        "end": (ORIGIN_DATE + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        "stride": 1,
        "tasks": [{
            "task_id": "wti_test",
            "target_series_id": WTI_SERIES_ID,
            "horizons": HORIZONS,
            "frequency": "B",
            "description": "WTI consistency test"
        }]
    })
    
    # Run backtest for this single origin
    backtest_result = cached_multi_backtest(
        predictor, spec, data_service,
        force_refresh=True,  # Always rerun for consistency test
    )
    
    results.append({
        "run_num": run_num + 1,
        "backtest_result": backtest_result,
        "predictor": predictor,
    })
    
    # Extract predictions from result
    for result_dict in backtest_result.values():
        for pred in result_dict.predictions:
            if isinstance(pred.payload, ContinuousForecast):
                print(f"  point=${pred.payload.point_forecast:.2f}")

print(f"\n✓ All {NUM_RUNS} runs complete")


RUN 1 / 3


/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


  point=$98.50
  point=$101.00
  point=$104.00
  point=$98.50
  point=$100.00
  point=$95.00

RUN 2 / 3
  point=$97.50
  point=$99.00
  point=$101.00
  point=$98.50
  point=$97.00
  point=$95.00

RUN 3 / 3
  point=$98.50
  point=$99.00
  point=$101.50
  point=$98.50
  point=$99.00
  point=$95.00

✓ All 3 runs complete


In [3]:
import inspect
from aieng.forecasting.data.context import ForecastContext

sig = inspect.signature(ForecastContext)
print(sig)

(store: 'SeriesStore', as_of: 'datetime', doc_store: 'DocumentStore | None' = None) -> 'None'


## Compare Point Forecasts Across Runs

In [4]:
# Extract point forecasts for each horizon across all runs
forecast_comparison = {h: [] for h in HORIZONS}

for run_data in results:
    backtest_result = run_data["backtest_result"]
    
    # backtest_result is a dict of results; extract from the first one
    for result in backtest_result.values():
        for pred in result.predictions:
            if isinstance(pred.payload, ContinuousForecast):
                # Find which horizon this is
                as_of = pd.Timestamp(pred.as_of)
                forecast_date = pd.Timestamp(pred.forecast_date)
                offset = pd.tseries.offsets.BDay()
                
                for h in HORIZONS:
                    target_date = as_of + offset * h
                    if target_date.normalize() == forecast_date.normalize():
                        forecast_comparison[h].append(pred.payload.point_forecast)
                        break

# Display comparison table
comparison_rows = []
for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if forecasts:
        row = {
            "Horizon": f"{h}d",
            "Run 1": f"${forecasts[0]:.2f}" if len(forecasts) > 0 else "—",
            "Run 2": f"${forecasts[1]:.2f}" if len(forecasts) > 1 else "—",
            "Run 3": f"${forecasts[2]:.2f}" if len(forecasts) > 2 else "—",
            "Mean": f"${np.mean(forecasts):.2f}",
            "Std Dev": f"${np.std(forecasts):.2f}",
            "Range": f"${np.max(forecasts) - np.min(forecasts):.2f}",
        }
        comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)
print("\n" + "="*72)
print("POINT FORECAST COMPARISON ACROSS 3 RUNS")
print("="*72)
print(df_comparison.to_string(index=False))


POINT FORECAST COMPARISON ACROSS 3 RUNS
Horizon   Run 1   Run 2   Run 3   Mean Std Dev Range
     5d  $98.50  $98.50  $97.50 $98.33   $0.37 $1.00
    10d $101.00 $100.00  $99.00 $99.17   $1.21 $4.00
    21d $104.00  $95.00 $101.00 $98.58   $3.70 $9.00


## Extract Full Distributions (Quantiles)

In [9]:
# Extract full distributions and build comparison table
all_distributions = {run_idx: {h: None for h in HORIZONS} for run_idx in range(NUM_RUNS)}

for run_idx, run_data in enumerate(results):
    backtest_result = run_data["backtest_result"]
    
    for result in backtest_result.values():
        for pred in result.predictions:
            if isinstance(pred.payload, ContinuousForecast):
                cf = pred.payload
                as_of = pd.Timestamp(pred.as_of)
                forecast_date = pd.Timestamp(pred.forecast_date)
                offset = pd.tseries.offsets.BDay()
                
                for h in HORIZONS:
                    target_date = as_of + offset * h
                    if target_date.normalize() == forecast_date.normalize():
                        if all_distributions[run_idx][h] is None:
                            all_distributions[run_idx][h] = cf
                        break

# Build table
dist_rows = []
for h in HORIZONS:
    row = {"Horizon": f"{h}d"}
    for run_idx in range(NUM_RUNS):
        cf = all_distributions[run_idx][h]
        if cf is not None:
            point = f"${cf.point_forecast:.2f}"
            
            # Try to get CI
            ci = ""
            if hasattr(cf, 'lower_quantile') and hasattr(cf, 'upper_quantile'):
                ci = f" [{cf.lower_quantile:.2f}, {cf.upper_quantile:.2f}]"
            elif hasattr(cf, 'quantile_forecasts') and cf.quantile_forecasts:
                q10 = cf.quantile_forecasts.get(0.1, "—")
                q90 = cf.quantile_forecasts.get(0.9, "—")
                ci = f" [{q10:.2f}, {q90:.2f}]"
            
            row[f"Run {run_idx+1}"] = point + ci
    dist_rows.append(row)

df_distributions = pd.DataFrame(dist_rows)
print("\n" + "="*72)
print("FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)")
print("="*72)
print(df_distributions.to_string(index=False))


FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)
Horizon   Run 1   Run 2   Run 3
     5d  $98.50  $97.50  $98.50
    10d $101.00  $99.00  $99.00
    21d $104.00 $101.00 $101.50


## Extract Rationales (Agent Reasoning)

In [10]:
# Extract rationales from prediction metadata - deduplicated
for run_idx, run_data in enumerate(results):
    print(f"\n{'='*72}")
    print(f"RUN {run_idx + 1} — AGENT RATIONALE")
    print(f"{'='*72}\n")
    
    backtest_result = run_data["backtest_result"]
    seen_rationales = set()
    
    # Extract from the first result in the dict
    for result in backtest_result.values():
        for pred in result.predictions:
            # Check for rationale in metadata
            if hasattr(pred, 'metadata') and pred.metadata:
                rationale = pred.metadata.get('rationale', None)
                if rationale and rationale not in seen_rationales:
                    print(rationale)
                    print()
                    seen_rationales.add(rationale)
            
            # Also check payload for any reasoning field
            if hasattr(pred.payload, 'reasoning'):
                reasoning = pred.payload.reasoning
                if reasoning and reasoning not in seen_rationales:
                    print(reasoning)
                    print()
                    seen_rationales.add(reasoning)


RUN 1 — AGENT RATIONALE

The market is dominated by the binary nature of the Strait of Hormuz situation. Volatility is extreme because the market must price both a total supply failure and a resolution. The forecast reflects this wide uncertainty, with skew skewed slightly toward the upside as the market is forced to hold a risk premium while the situation remains unresolved.

The market is currently dominated by the high geopolitical risk premium stemming from the Strait of Hormuz crisis. The core fundamental struggle is between physically tight inventories and the looming demand destruction caused by sustained high energy prices. The scenarios reflect the binary nature of the geopolitical risk.


RUN 2 — AGENT RATIONALE

The market is fundamentally driven by the severe geopolitical disruption at the Strait of Hormuz. My forecast assumes this persists, leading to elevated price levels, with scenarios primarily diverging based on whether the situation escalates, holds, or reaches a br

## Consistency Assessment

In [7]:
print("\n" + "="*72)
print("CONSISTENCY METRICS")
print("="*72)

for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if len(forecasts) == NUM_RUNS:
        mean = np.mean(forecasts)
        std = np.std(forecasts)
        cv = (std / mean) * 100
        print(f"\nh={h}d:")
        print(f"  Mean:                 ${mean:.2f}")
        print(f"  Std Dev:              ${std:.2f}")
        print(f"  Coefficient of Var:   {cv:.1f}%")
        print(f"  Range (max - min):    ${np.max(forecasts) - np.min(forecasts):.2f}")
        
        if cv < 1.0:
            consistency = "✓ Very consistent (CV < 1%)" 
        elif cv < 2.0:
            consistency = "✓ Consistent (CV 1-2%)"
        elif cv < 5.0:
            consistency = "⚠ Moderate variance (CV 2-5%)"
        else:
            consistency = "✗ High variance (CV > 5%)"
        print(f"  Assessment:           {consistency}")


CONSISTENCY METRICS


## Scenario Structure Comparison

**Scenarios identified in Run 1:**
- Scenario A: ...
- Scenario B: ...
- Scenario C: ...

**Scenarios identified in Run 2:**
- Scenario A: ...
- Scenario B: ...
- Scenario C: ...

**Scenarios identified in Run 3:**
- Scenario A: ...
- Scenario B: ...
- Scenario C: ...

**Consistency of scenarios:**
- Same three scenarios across all runs?
- Same base-case probability weighting?
- Same tail structure (upside vs downside)?